In [1]:
import os
import requests
from functools import partial
import time
from concurrent.futures import ThreadPoolExecutor

from pathlib import Path
parent_path = Path().absolute().parent
#all imported without issue with Python 3 (ipykernel)

Data Access Script - Standard Element 1 - Tuomas Haapala

No authentication protocol is required - the data is openly available.

Finnish Meteorological Institute offers gridded data for several meteorological variables. This project uses only precipitation.

In [2]:
GRIDDED_DATA_TO_DOWNLOAD = ["RRday/rrday_"] #Identifier for precipitation.
GRIDDED_DATA_URL = "https://fmi-gridded-obs-daily-1km.s3-eu-west-1.amazonaws.com/Netcdf/"
CHUNK_SIZE = 8192

This function creates the url list used to download gridded meteorological data from FMI.

In [3]:
def construct_gridded_urls(start_year, end_year):
    urls = []
    if end_year is None:
        end_year = start_year
    for var in GRIDDED_DATA_TO_DOWNLOAD:
        for year in range(start_year, end_year+1):
            urls.append([GRIDDED_DATA_URL + var + str(year) + ".nc"]) #"_months_4_to.nc" for PET
    return urls

This function downloads the gridded data file.

In [4]:
def download_file(url, save_path=None):
    try:
        local_filename = url.split('/')[-1]
        if save_path is not None:
            local_filename = fix_path(save_path) + local_filename

        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(local_filename, 'wb') as f:
                for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                    f.write(chunk)
    except Exception as e:
        print(e)

This function automatically fixes file paths with absent slashes.

In [5]:
def fix_path(path):
    if path[-1] != "/":
        path = path + "/"
    return path

This function uses the download_file function to go through urls and download the data into the data_folder.

In [6]:
def download_gridded_data(start_year, end_year):
    print("Downloading gridded data from FMI...")
    urls = construct_gridded_urls(start_year, end_year)
    download_func = partial(download_file, save_path=data_folder)
    start_downloading_time = time.time()
    with ThreadPoolExecutor() as executor:
        for i in urls:
            executor.map(download_func, i)
    end_downloading_time = time.time()
    print("Gridded data downloaded in {} s".format(end_downloading_time-start_downloading_time))

The main function creates the data folder directory in the same directory as the file, defines the start and end year of the dataset with user input, and begins the download process of the data.

In [7]:
def main():
    global start_year, end_year, data_folder
    """raw_data is created to the same folder where the script is saved."""
    file_location = str(parent_path) + "/inputs" 
    data_folder = fr"{file_location}" 
    try:
        os.makedirs(data_folder)
    except OSError as iex:
        print(f"Creation of directories failed! {iex}")
    
    "Program introduction and year inputs"
    print("This is the precipitation data downloader script. Data available for 1961-2025.")
    print("Note: minimum of three years required for full trend analysis.")
    #start_year = int(input("Please input the start year of the dataset: "))
    #end_year = int(input("Please input the end year of the dataset: "))
    #Default inputs:
    start_year = 1999
    end_year = 2001
    if start_year == end_year:
        print("One year of data to be downloaded.")
    else:
        print(end_year - start_year+1, "years of data to be downloaded.")
    #input("Press Enter to begin download:")
    download_gridded_data(start_year, end_year)
main()

Creation of directories failed! [Errno 17] File exists: '/home/jovyan/spi_event_trend_detector/inputs'
This is the precipitation data downloader script. Data available for 1961-2025.
Note: minimum of three years required for full trend analysis.
3 years of data to be downloaded.


Gridded data downloaded in 32.666961669921875 s
